In [10]:
#Set up

#install libraries
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from tueplots import bundles
from tueplots.constants.color import rgb
from tueplots.constants.color.palettes import paultol_vibrant 

# set plotting stylesheet
plt.rcParams.update(bundles.icml2024(column='half', nrows=1, ncols=1)) #test different rows and columns

import sys
print(sys.executable)

#set source and output paths
source_path = '../../data/'
csv_output_path = '../../data/processed/'
figure_output_path = '../../paper/figures/'

#upload raw query data
df_32_raw = pd.read_csv(f'{source_path}repository_queries/500000_32_homicide-female_DE.csv') 
final = pd.read_csv(f'{source_path}final_dataset_t225.csv')

/Users/madelinemiller/Desktop/data_literacy/geonews_femicide/source/.venv/bin/python


In [64]:

#filter to only date in 2019
#Convert the date column to datetime objects
final['date'] = pd.to_datetime(final['date'])      
# Filter for date
final_2019 = final[final['date'].dt.year == 2019].copy()
final_2018 = final[final['date'].dt.year == 2018].copy()
final_2020 = final[final['date'].dt.year == 2020].copy()

final_2019_DE915 = final_2019[
    final_2019['NUTS'].fillna('').str.split(',').apply(
        lambda x: 'Göttingen' in [v.strip() for v in x]
    )
]
print(final_2019.shape)
print(final_2019_DE915.shape)

(5789, 8)
(0, 8)


In [62]:
final_2019_ids = (
    df_32_raw.loc[df_32_raw['id'].isin(final_2019['id']), 'id']
    .unique()
    .tolist()
)

final_2018_ids = (
    df_32_raw.loc[df_32_raw['id'].isin(final_2018['id']), 'id']
    .unique()
    .tolist()
)

final_2020_ids = (
    df_32_raw.loc[df_32_raw['id'].isin(final_2020['id']), 'id']
    .unique()
    .tolist()
)


final_2019['year'] = 2019
final_2018['year'] = 2018
final_2020['year'] = 2020

#combine
final_all_years = pd.concat(
    [final_2018, final_2019, final_2020],
    ignore_index=True
)

print(final_all_years.shape)

merged_df = df_32_raw.merge(
    final_all_years[['id', 'year']],  # only need id and year from final_all_years
    on='id',
    how='left'                        # keeps all rows in df_32_raw
)

print(merged_df.shape)

nuts_article_counts = (
    merged_df
    .groupby(['NUTS', 'year'])['id']
    .nunique()
    .reset_index(name='unique_article_count')
)

nuts_pivot = nuts_article_counts.pivot(
    index='NUTS',
    columns='year',
    values='unique_article_count'
).reset_index()

nuts_pivot['diff'] = nuts_pivot[2019] - nuts_pivot[2018]
avg_diff = nuts_pivot['diff'].abs().mean()
print(f"Average difference 2018-2019: {avg_diff}")
std_dev = nuts_pivot['diff'].abs().std()
print(f"STD 2018-2019: {std_dev}")

nuts_pivot['flag_largeincrease'] = nuts_pivot['diff'] > std_dev + avg_diff 

nuts_pivot[nuts_pivot['flag_largeincrease']==True].sort_values('diff', ascending = False)

(13908, 9)
(1143913, 14)
Average difference 2018-2019: 36.33482142857143
STD 2018-2019: 71.24422369451041


year,NUTS,2018.0,2019.0,2020.0,diff,flag_largeincrease
184,DE91C,21.0,674.0,148.0,653.0,True
187,DE925,83.0,401.0,49.0,318.0,True
230,DEA1F,9.0,279.0,89.0,270.0,True
173,DE80N,11.0,277.0,5.0,266.0,True
171,DE80L,6.0,215.0,5.0,209.0,True
217,DEA12,10.0,213.0,70.0,203.0,True
144,DE712,218.0,407.0,85.0,189.0,True
191,DE929,321.0,473.0,184.0,152.0,True
162,DE731,5.0,152.0,5.0,147.0,True
145,DE713,26.0,167.0,30.0,141.0,True


In [43]:
loc_check.head()

,id,url,hostname,date,hashed_id,date_crawled,loc_normal,latitude,longitude,NUTS,query_string,query_name,cos_dist
25,4d9deab2-ca96-432a-8a7c-3c4d77a9b0fb,https://www.welt.de/regionales/rheinland-pfalz...,welt.de,2019-11-23,1004015002149151694,2019-11-23 00:00:00,mainz,49.992862,8.247253,DEB35,Tötungsdelikt mit weiblichem Opfer,32_homicide-female_DE,0.214758
26,4d9deab2-ca96-432a-8a7c-3c4d77a9b0fb,https://www.welt.de/regionales/rheinland-pfalz...,welt.de,2019-11-23,1004015002149151694,2019-11-23 00:00:00,worms,49.630262,8.362090,DEB39,Tötungsdelikt mit weiblichem Opfer,32_homicide-female_DE,0.214758
27,4d9deab2-ca96-432a-8a7c-3c4d77a9b0fb,https://www.welt.de/regionales/rheinland-pfalz...,welt.de,2019-11-23,1004015002149151694,2019-11-23 00:00:00,karlsruhe,49.006890,8.403653,DE122,Tötungsdelikt mit weiblichem Opfer,32_homicide-female_DE,0.214758
33,5bf1aaa2-68d2-4045-bb3f-f401dc49fdde,https://www.schwaebische.de/sueden/baden-wuert...,schwaebische.de,2019-02-19,1005047204544765696,2019-02-19 00:00:00,ehrenkirchen,47.922742,7.735158,DE132,Tötungsdelikt mit weiblichem Opfer,32_homicide-female_DE,0.219871
34,5bf1aaa2-68d2-4045-bb3f-f401dc49fdde,https://www.schwaebische.de/sueden/baden-wuert...,schwaebische.de,2019-02-19,1005047204544765696,2019-02-19 00:00:00,bad krozingen,47.911829,7.703331,DE132,Tötungsdelikt mit weiblichem Opfer,32_homicide-female_DE,0.219871


In [18]:
final_2019 = final[final['date']==2019]